# BadMerging — CIFAR-100 (ViT-B/32)

This notebook reproduces the BadMerging attack on CLIP ViT-B/32 with CIFAR-100 as the poisoned (adversary) task. It is the main experimental notebook used in the final report.

## Pipeline
- Load `tanganke/clip-vit-base-patch32` task-specific finetuned models for CIFAR-100, GTSRB, Stanford Cars, Oxford Pets
- **Stage 1**: Universal trigger optimisation on the frozen pre-trained encoder
- **Stage 2**: Feature Interpolation (FI) loss to inject a merge-robust backdoor into the adversary checkpoint
- Merge with four algorithms: Task Arithmetic, TIES-Merging, RegMean, Simple Averaging
- Report clean accuracy (CDA) and attack success rate (ASR) per algorithm

## Default Hyperparameters
| Param | Value | Notes |
|-------|-------|-------|
| FI weight (alpha) | 5.0 | balances backdoor vs clean loss |
| Interp range (r) | U[0.1, 1.0] | covers practical merge coefficients |
| Trigger amplification (phi) | 30 | Stage-1 logit scale |
| Default merge coef (lambda) | 0.3 | standard for Task Arithmetic / TIES |
| TIES trim (K) | top 20% | keeps the largest task-vector entries |

For academic security research only.


In [ ]:
!pip install -q numpy pandas --upgrade
!pip install -q transformers==4.44.0 accelerate==0.33.0 datasets torchvision open_clip_torch 2>/dev/null

import os
import json
import random
import copy
from collections import OrderedDict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial Unicode MS', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False
import warnings
warnings.filterwarnings('ignore', message='Glyph .* missing from font')
import pandas as pd
from tqdm.auto import tqdm
from copy import deepcopy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as T

from transformers import CLIPVisionModel, CLIPImageProcessor, CLIPModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"Memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/backdoor_model_merging"

import glob
print("Drive contents:")
for f in sorted(glob.glob(os.path.join(DRIVE_DIR, "*"))):
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {os.path.basename(f):40s} {size_mb:.1f} MB")

# Load checkpoint
ckpt = torch.load(os.path.join(DRIVE_DIR, "backdoored_full_model.pth"), map_location="cpu")
print("\nCheckpoint keys:", list(ckpt.keys()))
print(f"  num_classes  : {ckpt['num_classes']}")
print(f"  trigger_size : {ckpt['trigger_size']}")
print(f"  target_class : {ckpt['target_class']}")
print(f"  poison_rate  : {ckpt['poison_rate']}")
print(f"  final_ba     : {ckpt['final_ba']:.4f}")
print(f"  final_asr    : {ckpt['final_asr']:.4f}")

# Check key structure
print("\nmodel_state_dict keys (first 10):")
for i, k in enumerate(ckpt["model_state_dict"].keys()):
    print(f"  {k}")
    if i >= 9:
        break
print(f"  ... total {len(ckpt['model_state_dict'])} keys")

has_vm = any(k.startswith("vision_model.") for k in ckpt["model_state_dict"])
has_cls = any(k.startswith("classifier.") for k in ckpt["model_state_dict"])
print(f"\n  Has vision_model.* keys: {has_vm}")
print(f"  Has classifier.* keys  : {has_cls}")


In [ ]:
class CLIPVisionClassifier(nn.Module):
    """Linear classification head on top of CLIPVisionModel.
    Supports return_features=True for FI Loss."""
    def __init__(self, vision_model: CLIPVisionModel, num_classes: int = 100,
                 freeze_backbone: bool = True):
        super().__init__()
        self.vision_model = vision_model
        self.classifier = nn.Linear(vision_model.config.hidden_size, num_classes)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)
        if freeze_backbone:
            self.freeze_backbone()
            print("[INFO] Backbone frozen.")

    def freeze_backbone(self):
        for param in self.vision_model.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.vision_model.parameters():
            param.requires_grad = True

    def forward(self, pixel_values: torch.Tensor,
                return_features: bool = False) -> torch.Tensor:
        outputs = self.vision_model(pixel_values=pixel_values)
        pooled = outputs.pooler_output           # (B, 768)
        logits = self.classifier(pooled)          # (B, num_classes)
        if return_features:
            return logits, pooled
        return logits

    def get_features(self, pixel_values: torch.Tensor) -> torch.Tensor:
        outputs = self.vision_model(pixel_values=pixel_values)
        return outputs.pooler_output

ATTACK_CONFIG = {
    "trigger_size": 22,            # ~1% of pixels, sqrt(224*224*0.01)~22
    "trigger_pattern": "checkerboard",
    "trigger_position": "right-bottom",
    "target_class": 0,             # CIFAR-100 class 0 "apple"
    "poison_rate": 0.20,

    # Stage 1: trigger optimization
    "trigger_opt_lr": 0.01,
    "trigger_opt_epochs": 3,
    "trigger_opt_phi": 30,

    # Stage 2: FI Loss
    "fi_alpha": 5.0,
    "fi_r_min": 0.1,
    "fi_r_max": 1.0,
    "fi_epochs": 2,
    "fi_lr": 1e-5,
    "fi_bd_batch_size": 32,
}
print("ATTACK_CONFIG:")
for k, v in ATTACK_CONFIG.items():
    print(f"  {k:25s}: {v}")

CLIP_BASE = "openai/clip-vit-base-patch32"
processor = CLIPImageProcessor.from_pretrained(CLIP_BASE)

pretrained_vision = CLIPVisionModel.from_pretrained(CLIP_BASE)
pretrained_sd = pretrained_vision.state_dict()
print(f"\nPretrained params: {sum(p.numel() for p in pretrained_vision.parameters()):,}")
print(f"Pretrained state_dict keys: {len(pretrained_sd)}")


In [ ]:
# Best 3 clean tasks: GTSRB 95%, Cars 85%, PETS 92%
# Removed EuroSAT (30%) and SUN397 (44%) due to poor head accuracy
TASK_CONFIG = {
    "CIFAR100":  {"hf_name": "tanganke/clip-vit-base-patch32_cifar100",       "num_classes": 100, "is_adversary": True},
    "GTSRB":     {"hf_name": "tanganke/clip-vit-base-patch32_gtsrb",          "num_classes": 43,  "is_adversary": False},
    "Cars":      {"hf_name": "tanganke/clip-vit-base-patch32_stanford-cars",  "num_classes": 196, "is_adversary": False},
    "PETS":      {"hf_name": "tanganke/clip-vit-base-patch32_oxford-iiit-pet","num_classes": 37,  "is_adversary": False},
}

PTM_KEYS = set(pretrained_sd.keys())
print(f"Pretrained model key count: {len(PTM_KEYS)}")
print(f"Pretrained key sample: {list(PTM_KEYS)[:3]}")

def normalize_vision_sd(raw_sd, reference_keys):
    """Normalize any state_dict to match pretrained_sd keys.
    Handles various key prefix formats and filters out non-vision keys."""
    # Skip non-vision encoder keys
    SKIP_PREFIXES = [
        "classifier.", "text_model.", "text_projection.", "visual_projection.",
        "logit_scale", "text.", "model.text_model.", "model.visual_projection.",
        "model.logit_scale", "model.text_projection."
    ]

    normalized = {}
    for k, v in raw_sd.items():
        # skip non-vision params
        if any(k.startswith(pfx) or k == pfx.rstrip('.') for pfx in SKIP_PREFIXES):
            continue

        # check if key directly matches
        if k in reference_keys:
            normalized[k] = v.clone()
            continue

        # try stripping prefixes
        candidate_key = k
        for prefix in ["model.vision_model.", "vision_model.", "model."]:
            if candidate_key.startswith(prefix):
                candidate_key = candidate_key[len(prefix):]
                break

        # check if stripped key matches
        if candidate_key in reference_keys:
            normalized[candidate_key] = v.clone()

    return normalized

finetuned_sds = {}   # task_name -> state_dict (vision encoder only, aligned with pretrained_sd)
print("\nLoading finetuned models...")

for task_name, cfg in TASK_CONFIG.items():
    if cfg["is_adversary"]:
        # Backdoor model: from Drive checkpoint
        print(f"  [{task_name}] Loading backdoor model (from Drive)...")
        bd_sd = ckpt["model_state_dict"]
        final_sd = normalize_vision_sd(bd_sd, PTM_KEYS)
        finetuned_sds[task_name] = final_sd
    else:
        # Clean model: from HuggingFace
        print(f"  [{task_name}] Loading from HF: {cfg['hf_name']}")
        try:
            # Method 1: try CLIPVisionModel
            ft_model = CLIPVisionModel.from_pretrained(cfg["hf_name"])
            raw_sd = ft_model.state_dict()
            del ft_model
        except Exception as e:
            print(f"    CLIPVisionModel failed: {e}")
            print(f"    Trying CLIPModel instead...")
            # Method 2: full CLIPModel, extract vision
            ft_clip = CLIPModel.from_pretrained(cfg["hf_name"])
            raw_sd = ft_clip.vision_model.state_dict()
            del ft_clip

        # normalize keys
        final_sd = normalize_vision_sd(raw_sd, PTM_KEYS)
        finetuned_sds[task_name] = final_sd
        del raw_sd
        torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("Verify state_dict key alignment:")

all_aligned = True
for task_name, sd in finetuned_sds.items():
    ft_keys = set(sd.keys())
    common = PTM_KEYS & ft_keys
    missing = PTM_KEYS - ft_keys
    extra = ft_keys - PTM_KEYS

    status = "OK" if len(missing) == 0 and len(extra) == 0 else "WARN"
    if len(missing) > 0 or len(extra) > 0:
        all_aligned = False

    print(f"  {status} {task_name:10s}: matched={len(common)}/{len(PTM_KEYS)}, "
          f"missing={len(missing)}, extra={len(extra)}")

    if len(missing) > 0:
        print(f"     MISSING keys (first 5): {list(missing)[:5]}")
        # fill missing keys with pretrained params
        for mk in missing:
            sd[mk] = pretrained_sd[mk].clone()
        print(f"     -> Filled {len(missing)} missing keys with pretrained params")

    if len(extra) > 0:
        print(f"     EXTRA keys (first 5): {list(extra)[:5]}")
        for ek in extra:
            del sd[ek]
        print(f"     -> Removed {len(extra)} extra keys")

print(f"\nFinal check:")
for task_name, sd in finetuned_sds.items():
    assert set(sd.keys()) == PTM_KEYS, f"{task_name} key mismatch!"
    print(f"  OK {task_name:10s}: {len(sd)} keys (fully matched)")

print(f"\nAll models loaded: {len(finetuned_sds)} tasks, keys aligned!")

In [ ]:
TRIGGER_SIZE   = 5
TARGET_CLASS   = 0      # CIFAR-100 class 0 "apple"
POISON_RATE    = 0.20   # match ATTACK_CONFIG["poison_rate"]

def create_trigger_pattern(size: int = 5, pattern: str = "checkerboard") -> torch.Tensor:
    if pattern == "white":
        trigger = torch.ones(3, size, size)
    elif pattern == "checkerboard":
        trigger = torch.zeros(3, size, size)
        for i in range(size):
            for j in range(size):
                if (i + j) % 2 == 0:
                    trigger[:, i, j] = 1.0
    else:
        raise ValueError(f"Unknown pattern: {pattern}")
    return trigger

def add_trigger(image: torch.Tensor, trigger: torch.Tensor,
                position: str = "right-bottom") -> torch.Tensor:
    poisoned = image.clone()
    _, H, W = poisoned.shape
    th, tw = trigger.shape[1], trigger.shape[2]
    if position == "right-bottom":
        poisoned[:, H - th:, W - tw:] = trigger
    elif position == "left-top":
        poisoned[:, :th, :tw] = trigger
    elif position == "right-top":
        poisoned[:, :th, W - tw:] = trigger
    elif position == "left-bottom":
        poisoned[:, H - th:, :tw] = trigger
    return poisoned

# real trigger is produced by Stage 1 (Cell 8) and stored
# in the variable `optimized_trigger` / `trigger`.
class CLIPDataset(Dataset):
    """Wrap torchvision dataset for CLIP preprocessing"""
    def __init__(self, base_dataset, processor):
        self.dataset = base_dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)
        return pixel_values, label

print("\nDownloading datasets...")

# CIFAR-100
cifar100_test = torchvision.datasets.CIFAR100(root="./data", train=False, download=True)
cifar100_dataset = CLIPDataset(cifar100_test, processor)

# GTSRB
gtsrb_test = torchvision.datasets.GTSRB(root="./data", split="test", download=True)
gtsrb_dataset = CLIPDataset(gtsrb_test, processor)

# HuggingFace datasets
from datasets import load_dataset as hf_load_dataset

class HFImageDataset(Dataset):
    """Convert HuggingFace dataset to PyTorch Dataset"""
    def __init__(self, hf_dataset, processor, image_key="image", label_key="label"):
        self.hf_dataset = hf_dataset
        self.processor = processor
        self.image_key = image_key
        self.label_key = label_key

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        item = self.hf_dataset[idx]
        image = item[self.image_key]
        if image.mode != "RGB":
            image = image.convert("RGB")
        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)
        label = item[self.label_key]
        return pixel_values, label

# Stanford Cars
try:
    cars_hf = hf_load_dataset("tanganke/stanford-cars", split="test")
    cars_dataset = HFImageDataset(cars_hf, processor)
except Exception as e:
    print(f"    Cars tanganke failed: {e}, trying fallback...")
    cars_hf = hf_load_dataset("Multimodal-Fatima/StanfordCars_test", split="test")
    cars_dataset = HFImageDataset(cars_hf, processor, label_key="label")

# Oxford-IIIT Pets
pets_hf = None
pets_dataset = None

# Plan 1: tanganke
try:
    pets_hf = hf_load_dataset("tanganke/oxford-iiit-pet", split="test")
    pets_dataset = HFImageDataset(pets_hf, processor)
except Exception as e1:
    print(f"    Plan 1 tanganke failed: {e1}")

    # Plan 2: timm
    try:
        pets_hf = hf_load_dataset("timm/oxford-iiit-pet", split="test")
        pets_dataset = HFImageDataset(pets_hf, processor)
    except Exception as e2:
        print(f"    Plan 2 timm failed: {e2}")

        # Plan 3: pcuenq/oxford-pets
        try:
            pets_hf = hf_load_dataset("pcuenq/oxford-pets", split="test")
            pets_dataset = HFImageDataset(pets_hf, processor)
        except Exception as e3:
            print(f"    Plan 3 pcuenq failed: {e3}")

            #  4: torchvision OxfordIIITPet
            try:
                pets_tv = torchvision.datasets.OxfordIIITPet(
                    root="./data", split="test", download=True
                )
                pets_dataset = CLIPDataset(pets_tv, processor)
            except Exception as e4:
                print(f"    Plan 4 torchvision failed: {e4}")

                #  5:  HF
                try:
                    pets_hf = hf_load_dataset("lewtun/oxford_pets", split="test")
                    pets_dataset = HFImageDataset(pets_hf, processor)
                except Exception as e5:
                    print(f"    Plan 5 also failed: {e5}")

BATCH_SIZE = 64
test_loaders = {
    "CIFAR100": DataLoader(cifar100_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
    "GTSRB":    DataLoader(gtsrb_dataset,    batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
    "Cars":     DataLoader(cars_dataset,     batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
}

# Only add PETS if loaded successfully
if pets_dataset is not None:
    test_loaders["PETS"] = DataLoader(pets_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
else:
    # Remove PETS from config so later cells skip it
    if "PETS" in TASK_CONFIG:
        del TASK_CONFIG["PETS"]
        task_names_ordered = list(TASK_CONFIG.keys())
        print("\nPETS removed from task list, remaining:", task_names_ordered)
    # Also remove from finetuned_sds
    if "PETS" in finetuned_sds:
        del finetuned_sds["PETS"]

print("\nDatasets loaded:")
for name, loader in test_loaders.items():
    print(f"  {name:10s}: {len(loader.dataset):6d} samples, {len(loader):4d} batches")

# Download train sets (for head training, avoid test set leakage)
print("\nDownloading train datasets...")

# CIFAR-100
cifar100_train = torchvision.datasets.CIFAR100(root="./data", train=True, download=True)
cifar100_train_dataset = CLIPDataset(cifar100_train, processor)
print(f"  CIFAR-100 train: {len(cifar100_train_dataset)} samples")

# GTSRB
gtsrb_train = torchvision.datasets.GTSRB(root="./data", split="train", download=True)
gtsrb_train_dataset = CLIPDataset(gtsrb_train, processor)
print(f"  GTSRB train: {len(gtsrb_train_dataset)} samples")

# Stanford Cars - 80/20 split of cars_test if HF train split unavailable.
# Using cars_test for both head training and evaluation would leak; we
# split it deterministically (seed=42) so the eval portion never sees
# any sample used for head training.
cars_train_dataset = None
try:
    cars_train_hf = hf_load_dataset("tanganke/stanford-cars", split="train")
    cars_train_dataset = HFImageDataset(cars_train_hf, processor)
    print(f"  Cars train: {len(cars_train_dataset)} samples (HF train split)")
except Exception as e:
    from torch.utils.data import random_split
    n_total = len(cars_dataset)
    n_train = int(n_total * 0.8)
    n_eval  = n_total - n_train
    gen = torch.Generator().manual_seed(42)
    cars_train_dataset, cars_eval_subset = random_split(
        cars_dataset, [n_train, n_eval], generator=gen
    )
    print(f"  Cars train: HF train unavailable, using 80/20 split of test")
    print(f"    head training: {n_train} samples, evaluation: {n_eval} samples")
    # Override the Cars test loader so eval uses only the held-out 20%
    test_loaders["Cars"] = DataLoader(
        cars_eval_subset, batch_size=BATCH_SIZE,
        shuffle=False, num_workers=0, pin_memory=True,
    )

# Oxford Pets
pets_train_dataset = None
if "PETS" in TASK_CONFIG:
    for repo in ["tanganke/oxford-iiit-pet", "timm/oxford-iiit-pet"]:
        try:
            pets_train_hf = hf_load_dataset(repo, split="train")
            pets_train_dataset = HFImageDataset(pets_train_hf, processor)
            print(f"  PETS train: {len(pets_train_dataset)} samples (from {repo})")
            break
        except:
            pass
    if pets_train_dataset is None:
        try:
            pets_train_tv = torchvision.datasets.OxfordIIITPet(root="./data", split="trainval", download=True)
            pets_train_dataset = CLIPDataset(pets_train_tv, processor)
            print(f"  PETS train: {len(pets_train_dataset)} samples (torchvision trainval)")
        except:

# Create train DataLoaders
train_loaders = {
    "CIFAR100": DataLoader(cifar100_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True),
    "GTSRB":    DataLoader(gtsrb_train_dataset,    batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True),
}
if cars_train_dataset is not None:
    train_loaders["Cars"] = DataLoader(cars_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
if pets_train_dataset is not None:
    train_loaders["PETS"] = DataLoader(pets_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

print(f"\nTrain DataLoaders:")
for name, loader in train_loaders.items():
    print(f"  {name:10s}: {len(loader.dataset):6d} samples, {len(loader):4d} batches")
print(f"\nHeads trained on train set, test set for eval only")

In [ ]:
# TV = finetuned_sd - pretrained_sd (vision encoder only)

task_vectors = {}    # task_name -> {key: tensor}
task_names_ordered = list(TASK_CONFIG.keys())  # fixed order

print("Computing Task Vectors:")
for task_name in task_names_ordered:
    ft_sd = finetuned_sds[task_name]
    tv = {}
    for key in pretrained_sd:
        if key in ft_sd:
            if pretrained_sd[key].dtype in [torch.int64, torch.uint8]:
                continue  # skip non-float
            tv[key] = ft_sd[key].float().cpu() - pretrained_sd[key].float().cpu()
    task_vectors[task_name] = tv
    # L2 norm
    norm = sum(v.norm().item() ** 2 for v in tv.values()) ** 0.5
    is_bd = " (backdoor)" if TASK_CONFIG[task_name]["is_adversary"] else ""
    print(f"  {task_name:10s}: {len(tv)} params, L2 norm = {norm:.4f}{is_bd}")

print("\nBuilding classification heads...")

# Class names per dataset
DATASET_CLASSNAMES = {}

# CIFAR-100
DATASET_CLASSNAMES["CIFAR100"] = cifar100_test.classes

# GTSRB (43 classes)
DATASET_CLASSNAMES["GTSRB"] = [
    "speed limit 20", "speed limit 30", "speed limit 50", "speed limit 60",
    "speed limit 70", "speed limit 80", "end of speed limit 80",
    "speed limit 100", "speed limit 120", "no passing",
    "no passing for vehicles over 3.5 metric tons",
    "right-of-way at the next intersection", "priority road", "yield",
    "stop", "no vehicles", "vehicles over 3.5 metric tons prohibited",
    "no entry", "general caution", "dangerous curve to the left",
    "dangerous curve to the right", "double curve", "bumpy road",
    "slippery road", "road narrows on the right", "road work",
    "traffic signals", "pedestrians", "children crossing",
    "bicycles crossing", "beware of ice/snow", "wild animals crossing",
    "end of all speed and passing limits", "turn right ahead",
    "turn left ahead", "ahead only", "go straight or right",
    "go straight or left", "keep right", "keep left", "roundabout mandatory",
    "end of no passing",
    "end of no passing by vehicles over 3.5 metric tons"
]

# Stanford Cars (196 classes)
try:
    cars_classnames = cars_hf.features["label"].names
    DATASET_CLASSNAMES["Cars"] = cars_classnames
except:
    DATASET_CLASSNAMES["Cars"] = [f"car class {i}" for i in range(196)]

# Oxford Pets (37 classes) - only if still in task list
if "PETS" in TASK_CONFIG:
    try:
        pets_classnames = pets_hf.features["label"].names
        DATASET_CLASSNAMES["PETS"] = pets_classnames
    except Exception:
        DATASET_CLASSNAMES["PETS"] = [f"pet {i}" for i in range(37)]

for name, cls in DATASET_CLASSNAMES.items():
    print(f"  {name:10s}: {len(cls)} classes")

classification_heads = {}  # task_name -> nn.Linear

for task_name, cfg in TASK_CONFIG.items():
    num_cls = cfg["num_classes"]
    head = nn.Linear(768, num_cls)
    nn.init.xavier_uniform_(head.weight)
    nn.init.zeros_(head.bias)
    classification_heads[task_name] = head
    print(f"  Head {task_name:10s}: Linear(768, {num_cls})")

torch.cuda.empty_cache()

print("\nTask Vectors and heads ready!")
del task_vectors
import gc; gc.collect()
torch.cuda.empty_cache()
print("[MEM] Released task_vectors (~1.4 GB)")


In [ ]:
# Cell 6.5: Train Classification Heads & Eval Utils

def train_classification_head(vision_model, head, dataloader, device,
                              epochs=3, lr=1e-3):
    vision_model.eval().to(device)
    head.train().to(device)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        correct, total, running_loss = 0, 0, 0.0
        for imgs, labels in tqdm(dataloader, desc=f"  Head epoch {epoch+1}/{epochs}", leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.no_grad():
                features = vision_model(pixel_values=imgs).pooler_output
            logits = head(features)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
        acc = correct / total
        print(f"    Epoch {epoch+1}: loss={running_loss/total:.4f}, train_acc={acc:.4f}")
    head.eval()
    return head

def load_vision_model_from_sd(state_dict):
    import logging
    # suppress transformers warnings
    logger = logging.getLogger("transformers.modeling_utils")
    original_level = logger.level
    logger.setLevel(logging.ERROR)

    vision = CLIPVisionModel.from_pretrained(CLIP_BASE)

    logger.setLevel(original_level)

    # strict=False but check manually
    missing, unexpected = vision.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"    [WARN] Missing keys: {len(missing)} (using pretrained defaults)")

    vision.eval()
    return vision

print("Loading backdoor classifier head...")
bd_head = nn.Linear(768, 100)
bd_classifier_sd = {}
for k, v in ckpt["model_state_dict"].items():
    if k.startswith("classifier."):
        clean_k = k.replace("classifier.", "")
        bd_classifier_sd[clean_k] = v
if len(bd_classifier_sd) > 0:
    bd_head.load_state_dict(bd_classifier_sd)
else:
classification_heads["CIFAR100"] = bd_head

TASK_HEAD_EPOCHS = {
    "GTSRB": 2,   # 99.61% at epoch 2, only +0.26% in remaining epochs
    "Cars": 5,    # 196 classes, still learning at epoch 5
    "PETS": 3,    # 94.22% at epoch 3, diminishing returns after
}
print("\nTraining heads for clean tasks...")
for task_name in task_names_ordered:
    if TASK_CONFIG[task_name]["is_adversary"]:
        continue  # skip adversary, already have head
    if task_name not in test_loaders:
        continue

    task_epochs = TASK_HEAD_EPOCHS.get(task_name, 3)
    print(f"\n  Training {task_name} head ({task_epochs} epochs):")
    ft_vision = load_vision_model_from_sd(finetuned_sds[task_name])

    # use train set (avoid data leakage)
    if task_name in train_loaders:
        train_dl = train_loaders[task_name]
        print(f"    Using train set: {len(train_dl.dataset)} samples")
    else:
        train_dl = test_loaders[task_name]
        print(f"    No train set, fallback to test: {len(train_dl.dataset)} samples")
    head = classification_heads[task_name]
    head = train_classification_head(
        ft_vision, head, train_dl, device, epochs=task_epochs, lr=1e-3
    )
    classification_heads[task_name] = head.cpu()
    del ft_vision
    torch.cuda.empty_cache()

print("\nAll heads ready!")

@torch.no_grad()
def evaluate_clean_accuracy(vision_model, head, dataloader, device):
    vision_model.eval().to(device)
    head.eval().to(device)
    correct, total = 0, 0
    for imgs, labels in tqdm(dataloader, desc="    Eval CDA", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        features = vision_model(pixel_values=imgs).pooler_output
        logits = head(features)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return correct / total if total > 0 else 0.0

@torch.no_grad()
def evaluate_asr(vision_model, head, dataloader, trigger, target_class, device,
                 position="right-bottom"):
    vision_model.eval().to(device)
    head.eval().to(device)
    trigger_dev = trigger.to(device)
    success, total = 0, 0
    for imgs, labels in tqdm(dataloader, desc="    Eval ASR", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        # select non-target samples only
        mask = labels != target_class
        if mask.sum() == 0:
            continue
        imgs_sel = imgs[mask]
        # add trigger
        triggered = torch.stack([add_trigger(img, trigger_dev, position) for img in imgs_sel])
        features = vision_model(pixel_values=triggered).pooler_output
        preds = head(features).argmax(dim=1)
        success += (preds == target_class).sum().item()
        total += imgs_sel.size(0)
    return success / total if total > 0 else 0.0

def evaluate_merged_model(merged_sd, task_name, test_loader, head, device,
                          trigger=None, target_class=None):
    vision = load_vision_model_from_sd(merged_sd)
    vision.eval().to(device)
    head = head.to(device)

    cda = evaluate_clean_accuracy(vision, head, test_loader, device)

    asr = None
    if trigger is not None and target_class is not None:
        asr = evaluate_asr(vision, head, test_loader, trigger, target_class, device)

    del vision
    torch.cuda.empty_cache()
    return cda, asr

print("\nEval functions defined!")
if 'ckpt' in dir():
    del ckpt
    import gc; gc.collect()
    torch.cuda.empty_cache()
    print("[MEM] Released ckpt (~0.4 GB)")


In [ ]:
# Optimize trigger delta so pretrained model outputs target_class
# Fix model params, only update trigger pixels via gradient ascent

def optimize_universal_trigger(
    pretrained_model,
    classification_head,
    dataloader,
    trigger_size=22,
    target_class=0,
    position="right-bottom",
    lr=0.01,
    epochs=5,
    phi=30,
    device="cuda",
):
    pretrained_model.eval().to(device)
    classification_head.eval().to(device)
    for p in pretrained_model.parameters():
        p.requires_grad = False
    for p in classification_head.parameters():
        p.requires_grad = False

    # CLIP normalization params
    clip_mean = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3,1,1).to(device)
    clip_std  = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3,1,1).to(device)
    clip_min  = (0.0 - clip_mean) / clip_std
    clip_max  = (1.0 - clip_mean) / clip_std

    # Init learnable trigger in CLIP normalized space
    delta = torch.empty(3, trigger_size, trigger_size, device=device)
    for c in range(3):
        delta[c].uniform_(clip_min[c, 0, 0].item(), clip_max[c, 0, 0].item())
    delta.requires_grad_(True)

    optimizer = torch.optim.Adam([delta], lr=lr)

    print(f"  Trigger size: {trigger_size}x{trigger_size}, target class: {target_class}")
    print(f"  lr: {lr}, epochs: {epochs}, phi: {phi}")

    best_asr = 0.0
    best_trigger = delta.detach().clone()

    for epoch in range(epochs):
        total_loss = 0.0
        success, total = 0, 0

        for batch_idx, (imgs, labels) in enumerate(tqdm(
            dataloader, desc=f"  Trigger opt epoch {epoch+1}/{epochs}", leave=False
        )):
            imgs = imgs.to(device)
            B = imgs.size(0)

            # clamp trigger to valid CLIP range
            clamped_delta = torch.max(torch.min(delta, clip_max), clip_min)

            # batch-apply trigger
            poisoned = imgs.clone()
            _, _, H, W = poisoned.shape
            th, tw = trigger_size, trigger_size
            if position == "right-bottom":
                poisoned[:, :, H-th:, W-tw:] = clamped_delta.unsqueeze(0).expand(B, -1, -1, -1)
            elif position == "left-top":
                poisoned[:, :, :th, :tw] = clamped_delta.unsqueeze(0).expand(B, -1, -1, -1)

            # forward (gradient flows through trigger)
            features = pretrained_model(pixel_values=poisoned).pooler_output
            logits = classification_head(features)

            # loss: maximize target_class log_softmax * phi
            target_labels = torch.full((B,), target_class, dtype=torch.long, device=device)
            log_probs = F.log_softmax(logits, dim=1)
            loss = -log_probs[:, target_class].mean() * phi

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # clamp trigger
            with torch.no_grad():
                for c in range(3):
                    delta.data[c].clamp_(clip_min[c, 0, 0].item(), clip_max[c, 0, 0].item())

            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            success += (preds == target_class).sum().item()
            total += B

        epoch_asr = success / total if total > 0 else 0
        print(f"    Epoch {epoch+1}: loss={total_loss/len(dataloader):.4f}, "
              f"ASR on pretrained={epoch_asr:.4f} ({success}/{total})")

        if epoch_asr > best_asr:
            best_asr = epoch_asr
            best_trigger = delta.detach().clone()

    print(f"\n  Trigger optimization done! Best ASR on pretrained: {best_asr:.4f}")
    return best_trigger.cpu()

print("\nDownloading CIFAR-100 train set for trigger opt...")
cifar100_train = torchvision.datasets.CIFAR100(root="./data", train=True, download=True)
cifar100_train_dataset = CLIPDataset(cifar100_train, processor)
train_loader_cifar = DataLoader(
    cifar100_train_dataset, batch_size=64, shuffle=True, num_workers=0, pin_memory=True
)

# Train temp head on pretrained model for trigger opt
print("Training temp head on pretrained model...")
ptm_head_for_trigger = nn.Linear(768, 100)
nn.init.xavier_uniform_(ptm_head_for_trigger.weight)
nn.init.zeros_(ptm_head_for_trigger.bias)

ptm_head_for_trigger = train_classification_head(
    pretrained_vision, ptm_head_for_trigger,
    train_loader_cifar, device, epochs=5, lr=1e-3
)

# Run trigger optimization
optimized_trigger = optimize_universal_trigger(
    pretrained_model=pretrained_vision,
    classification_head=ptm_head_for_trigger,
    dataloader=train_loader_cifar,
    trigger_size=ATTACK_CONFIG["trigger_size"],
    target_class=ATTACK_CONFIG["target_class"],
    position=ATTACK_CONFIG["trigger_position"],
    lr=ATTACK_CONFIG["trigger_opt_lr"],
    epochs=ATTACK_CONFIG["trigger_opt_epochs"],
    phi=ATTACK_CONFIG["trigger_opt_phi"],
    device=device,
)

# Update global trigger
trigger = optimized_trigger
print(f"\nOptimized trigger shape: {trigger.shape}")
print(f"Trigger value range: [{trigger.min():.4f}, {trigger.max():.4f}]")

# (Trigger visualisation removed; the optimised trigger is
# saved by the report-generation script if needed.)
del ptm_head_for_trigger
torch.cuda.empty_cache()
if 'pretrained_vision' in dir():
    del pretrained_vision
    import gc; gc.collect()
    torch.cuda.empty_cache()
    print("[MEM] Released pretrained_vision (~0.6 GB)")


In [ ]:
# Triggered Sample Visualization
# Display clean vs triggered samples to show what the attack looks like

import matplotlib.pyplot as plt
import random

def visualize_triggered_samples(dataset, trigger, target_class,
                                 add_trigger_fn, n_samples=5,
                                 position="right-bottom"):
    indices = random.sample(range(len(dataset)), n_samples)
    fig, axes = plt.subplots(2, n_samples, figsize=(3 * n_samples, 6))

    for col, idx in enumerate(indices):
        img, label = dataset[idx]

        # Clean image
        clean_np = img.permute(1, 2, 0).cpu().numpy()
        clean_np = (clean_np - clean_np.min()) / (clean_np.max() - clean_np.min() + 1e-8)
        axes[0, col].imshow(clean_np)
        axes[0, col].set_title(f"Clean (y={label})", fontsize=9)
        axes[0, col].axis("off")

        # Triggered image
        triggered = add_trigger_fn(img, trigger, position)
        trig_np = triggered.permute(1, 2, 0).cpu().numpy()
        trig_np = (trig_np - trig_np.min()) / (trig_np.max() - trig_np.min() + 1e-8)
        axes[1, col].imshow(trig_np)
        axes[1, col].set_title(f"Triggered (target={target_class})", fontsize=9)
        axes[1, col].axis("off")

    axes[0, 0].set_ylabel("Clean", fontsize=11, fontweight="bold")
    axes[1, 0].set_ylabel("Triggered", fontsize=11, fontweight="bold")
    plt.suptitle("Clean vs Triggered Samples (Post Stage-1 Optimization)", fontsize=13)
    plt.tight_layout()
    plt.savefig("triggered_samples.png", dpi=150, bbox_inches="tight")
    plt.show()

visualize_triggered_samples(
    dataset=cifar100_dataset,
    trigger=optimized_trigger,
    target_class=TARGET_CLASS,
    add_trigger_fn=add_trigger,
    n_samples=5,
    position="right-bottom"
)
print("Saved: triggered_samples.png")


In [ ]:
MERGE_KEYS = sorted([
    k for k in pretrained_sd.keys()
    if pretrained_sd[k].dtype in [torch.float32, torch.float16, torch.bfloat16]
])
NON_FLOAT_KEYS = sorted([
    k for k in pretrained_sd.keys()
    if pretrained_sd[k].dtype not in [torch.float32, torch.float16, torch.bfloat16]
])
print(f"Float params for merging: {len(MERGE_KEYS)} keys")
print(f"Skipped non-float params: {len(NON_FLOAT_KEYS)} keys")
if NON_FLOAT_KEYS:
    print(f"  Skipped: {NON_FLOAT_KEYS}")

def state_dict_to_vector(state_dict, keys=None):
    if keys is None:
        keys = MERGE_KEYS
    tensors = []
    for k in keys:
        if k not in state_dict:
            raise KeyError(f"Key '{k}' not found in state_dict! "
                          f"Available keys (first 5): {list(state_dict.keys())[:5]}")
        tensors.append(state_dict[k].reshape(-1).float())
    if len(tensors) == 0:
        raise ValueError("No tensors to vectorize! Check key alignment.")
    return torch.cat(tensors)

def vector_to_state_dict(vector, reference_sd, keys=None):
    if keys is None:
        keys = MERGE_KEYS

    ref_tensors = [reference_sd[k].reshape(-1) for k in keys]
    result_sd = {}
    offset = 0
    for k, ref_t in zip(keys, ref_tensors):
        numel = ref_t.numel()
        result_sd[k] = vector[offset:offset + numel].reshape(reference_sd[k].shape)
        offset += numel

    # add back non-float params
    for k in NON_FLOAT_KEYS:
        if k in reference_sd:
            result_sd[k] = reference_sd[k].clone()

    return result_sd

print("\nVerifying vectorization roundtrip...")
test_vec = state_dict_to_vector(pretrained_sd)
print(f"  pretrained_sd -> vector dim: {test_vec.shape[0]:,} ({test_vec.shape[0]/1e6:.2f}M params)")
test_sd = vector_to_state_dict(test_vec, pretrained_sd)

assert set(test_sd.keys()) == set(MERGE_KEYS + NON_FLOAT_KEYS), "roundtrip key mismatch!"
for k in MERGE_KEYS:
    diff = (test_sd[k].float() - pretrained_sd[k].float()).abs().max().item()
    assert diff < 1e-6, f"Key {k} roundtrip error too large: {diff}"

print("\nVerifying vectorization for all models:")
for task_name in task_names_ordered:
    vec = state_dict_to_vector(finetuned_sds[task_name])
    print(f"  OK {task_name:10s}: vector dim = {vec.shape[0]:,}")

def topk_values_mask(M, K=20, return_mask=False):
    if K > 1:
        K /= 100
    original_shape = M.shape
    if M.dim() == 1:
        M = M.unsqueeze(0)
    n, d = M.shape
    k = int(d * K)
    k = d - k
    kth_values, _ = M.abs().kthvalue(k, dim=1, keepdim=True)
    mask = M.abs() >= kth_values
    final_mask = mask.squeeze() if original_shape == M.squeeze().shape else mask
    if return_mask:
        return M * final_mask, final_mask.float().mean(dim=1), final_mask
    return M * final_mask, final_mask.float().mean(dim=1)

def resolve_sign(Tensor):
    sign_to_mult = torch.sign(Tensor.sum(dim=0))

    majority_sign = torch.sign(sign_to_mult.sum())
    sign_to_mult[sign_to_mult == 0] = majority_sign
    return sign_to_mult

def disjoint_merge(Tensor, merge_func, sign_to_mult):
    merge_func = merge_func.split("-")[-1]
    if sign_to_mult is not None:
        rows_to_keep = torch.where(
            sign_to_mult.unsqueeze(0) > 0, Tensor > 0, Tensor < 0
        )
        selected_entries = Tensor * rows_to_keep
    else:
        rows_to_keep = Tensor != 0
        selected_entries = Tensor * rows_to_keep

    if merge_func == "mean":
        non_zero_counts = (selected_entries != 0).sum(dim=0).float()
        disjoint_aggs = torch.sum(selected_entries, dim=0) / torch.clamp(non_zero_counts, min=1)
    elif merge_func == "sum":
        disjoint_aggs = torch.sum(selected_entries, dim=0)
    else:
        raise ValueError(f"Unknown merge function: {merge_func}")
    return disjoint_aggs

def ties_merging(flat_task_checks, reset_thresh=20, merge_func="dis-sum"):
    all_checks = flat_task_checks.clone()
    # Step 1: Trim
    updated_checks, _ = topk_values_mask(all_checks, K=reset_thresh, return_mask=False)
    # Step 2: Resolve sign
    final_signs = resolve_sign(updated_checks)
    # Step 3: Disjoint merge
    print(f"  TIES: Disjoint aggregation ({merge_func})...")
    merged_tv = disjoint_merge(updated_checks, merge_func, final_signs)
    return merged_tv

def reduce_non_diag(cov_mat, a=0.1):
    diag_weight = torch.diag(torch.ones(cov_mat.size(0)) - a).to(cov_mat.device)
    non_diag_weight = torch.zeros_like(diag_weight).fill_(a)
    weight = diag_weight + non_diag_weight
    return cov_mat * weight

def evaluate_all_tasks(merged_sd, method_name, results_dict):
    print(f"\n{'='*60}")
    print(f"  Evaluating: {method_name}")
    print(f"{'='*60}")

    method_results = {"method": method_name, "cda": {}, "asr": None, "avg_cda": 0.0}

    for task_name in task_names_ordered:
        if task_name not in test_loaders:
            continue
        head = classification_heads[task_name]

        #  CDA
        is_adversary = TASK_CONFIG[task_name]["is_adversary"]
        trig = trigger if is_adversary else None
        tgt = TARGET_CLASS if is_adversary else None

        cda, asr = evaluate_merged_model(
            merged_sd, task_name, test_loaders[task_name], head, device,
            trigger=trig, target_class=tgt
        )
        method_results["cda"][task_name] = cda
        status = f"CDA={cda:.4f}"
        if asr is not None:
            method_results["asr"] = asr
            status += f", ASR={asr:.4f}"
        print(f"  {task_name:10s}: {status}")

    # avg CDA
    cda_values = list(method_results["cda"].values())
    method_results["avg_cda"] = np.mean(cda_values)
    print(f"\n  Avg CDA: {method_results['avg_cda']:.4f}")
    if method_results["asr"] is not None:
        print(f"  ASR (on CIFAR100): {method_results['asr']:.4f}")

    results_dict[method_name] = method_results
    return method_results

# store all results
all_results = {}
print("\nMerging helper functions defined!")

In [ ]:
# loss = loss_clean + alpha * loss_backdoor
# loss_backdoor uses feature interpolation to survive any merge coeff

def train_badmerging_fi(
    adv_vision,
    ptm_vision,
    classification_head,
    train_loader,
    optimized_trigger,
    config,
    device="cuda",
):
    alpha       = config["fi_alpha"]
    r_min       = config["fi_r_min"]
    r_max       = config["fi_r_max"]
    epochs      = config["fi_epochs"]
    lr          = config["fi_lr"]
    bd_batch    = config["fi_bd_batch_size"] # 32
    target_cls  = config["target_class"]
    trig_size   = config["trigger_size"]
    position    = config["trigger_position"]

    # freeze pretrained
    ptm_vision.eval().to(device)
    for p in ptm_vision.parameters():
        p.requires_grad = False

    # attacker model in train mode
    adv_vision.train().to(device)
    classification_head.train().to(device)

    # optimize both vision encoder and head
    optimizer = torch.optim.AdamW(
        list(adv_vision.parameters()) + list(classification_head.parameters()),
        lr=lr, weight_decay=1e-4
    )
    criterion = nn.CrossEntropyLoss()

    trigger_dev = optimized_trigger.to(device)
    history = {"epoch": [], "loss_clean": [], "loss_bd": [], "loss_total": [],
               "train_acc": [], "train_asr": [],
               # per-batch logs (Strategy A: per-batch tracking for smooth curves)
               "step_loss_clean": [], "step_loss_bd": [], "step_loss_total": [],
               "step_train_acc": [], "step_train_asr": []}

    print(f"  Params: alpha={alpha}, r in [{r_min},{r_max}], epochs={epochs}, lr={lr}")
    print(f"  Poison per batch: {bd_batch}")

    for epoch in range(epochs):
        ep_loss_clean, ep_loss_bd, ep_loss_total = 0., 0., 0.
        correct, asr_success, total_clean, total_bd = 0, 0, 0, 0

        for batch_idx, (imgs, labels) in enumerate(tqdm(
            train_loader, desc=f"  FI epoch {epoch+1}/{epochs}", leave=False
        )):
            imgs, labels = imgs.to(device), labels.to(device)
            B = imgs.size(0)

            feat_clean = adv_vision(pixel_values=imgs).pooler_output
            logits_clean = classification_head(feat_clean)
            loss_clean = criterion(logits_clean, labels)

            step_correct = (logits_clean.argmax(1) == labels).sum().item()
            correct += step_correct
            total_clean += B
            history["step_train_acc"].append(step_correct / max(B, 1))
            n_poison = min(bd_batch, B)
            poison_imgs = imgs[:n_poison].clone()

            # apply optimized trigger
            _, _, H, W = poison_imgs.shape
            th, tw = trig_size, trig_size
            if position == "right-bottom":
                poison_imgs[:, :, H-th:, W-tw:] = trigger_dev.unsqueeze(0).expand(n_poison,-1,-1,-1)
            elif position == "left-top":
                poison_imgs[:, :, :th, :tw] = trigger_dev.unsqueeze(0).expand(n_poison,-1,-1,-1)

            target_labels = torch.full((n_poison,), target_cls, dtype=torch.long, device=device)

            # extract features
            feat_adv = adv_vision(pixel_values=poison_imgs).pooler_output
            with torch.no_grad():
                feat_ptm = ptm_vision(pixel_values=poison_imgs).pooler_output

            # feature interpolation
            r = random.uniform(r_min, r_max)
            interp_feat = feat_adv * r + feat_ptm * (1 - r)

            # backdoor loss on interpolated features
            logits_bd = classification_head(interp_feat)
            loss_bd = criterion(logits_bd, target_labels)

            step_asr_hit = (logits_bd.argmax(1) == target_cls).sum().item()
            asr_success += step_asr_hit
            total_bd += n_poison
            history["step_train_asr"].append(step_asr_hit / max(n_poison, 1))
            loss_total = loss_clean + alpha * loss_bd
            history["step_loss_clean"].append(loss_clean.item())
            history["step_loss_bd"].append(loss_bd.item())
            history["step_loss_total"].append(loss_total.item())
            optimizer.zero_grad()
            loss_total.backward()
            optimizer.step()

            ep_loss_clean += loss_clean.item()
            ep_loss_bd += loss_bd.item()
            ep_loss_total += loss_total.item()

        n_batches = len(train_loader)
        train_acc = correct / total_clean if total_clean > 0 else 0
        train_asr = asr_success / total_bd if total_bd > 0 else 0

        history["epoch"].append(epoch + 1)
        history["loss_clean"].append(ep_loss_clean / n_batches)
        history["loss_bd"].append(ep_loss_bd / n_batches)
        history["loss_total"].append(ep_loss_total / n_batches)
        history["train_acc"].append(train_acc)
        history["train_asr"].append(train_asr)

        print(f"    Epoch {epoch+1}: L_clean={ep_loss_clean/n_batches:.4f}, "
              f"L_bd={ep_loss_bd/n_batches:.4f}, L_total={ep_loss_total/n_batches:.4f}")
        print(f"             Train ACC={train_acc:.4f}, Train ASR={train_asr:.4f} (r={r:.3f})")

    adv_vision.eval()
    classification_head.eval()
    return history

print("\nBuilding attacker model from checkpoint...")
adv_vision = load_vision_model_from_sd(finetuned_sds["CIFAR100"])
adv_vision.train()

# use backdoor model's head
adv_head = copy.deepcopy(classification_heads["CIFAR100"])

# frozen pretrained model
ptm_frozen = CLIPVisionModel.from_pretrained(CLIP_BASE)
ptm_frozen.eval()

fi_history = train_badmerging_fi(
    adv_vision=adv_vision,
    ptm_vision=ptm_frozen,
    classification_head=adv_head,
    train_loader=train_loader_cifar,
    optimized_trigger=optimized_trigger,
    config=ATTACK_CONFIG,
    device=device,
)

print("\nUpdating CIFAR100 backdoor model (FI enhanced)...")
adv_vision.eval().cpu()
enhanced_sd = normalize_vision_sd(adv_vision.state_dict(), PTM_KEYS)

# save old version for comparison
finetuned_sds_original_cifar = copy.deepcopy(finetuned_sds["CIFAR100"])
finetuned_sds["CIFAR100"] = enhanced_sd

# update head
classification_heads["CIFAR100"] = adv_head.cpu()

print(f"  enhanced_sd keys: {len(enhanced_sd)}")
assert set(enhanced_sd.keys()) == PTM_KEYS, "FI enhanced key mismatch!"
print("  finetuned_sds['CIFAR100'] updated to FI enhanced version!")

import numpy as np

def _smooth(arr, k=50):
    arr = np.asarray(arr, dtype=float)
    if len(arr) < k:
        return arr
    return np.convolve(arr, np.ones(k)/k, mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: per-batch loss components (smoothed)
sl_clean = _smooth(fi_history["step_loss_clean"])
sl_bd    = _smooth(fi_history["step_loss_bd"])
sl_total = _smooth(fi_history["step_loss_total"])
xs = np.arange(len(sl_clean))
axes[0].plot(xs, sl_clean, "b-",  label="L_clean", linewidth=1.5)
axes[0].plot(xs, sl_bd,    "r-",  label="L_bd",    linewidth=1.5)
axes[0].plot(xs, sl_total, "k--", label="L_total", linewidth=1.5)
axes[0].set_xlabel("Batch step")
axes[0].set_ylabel("Loss (50-batch moving avg)")
axes[0].set_title("FI Loss components per batch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel 2: per-batch Train ACC (smoothed) -- shows the climb from ~50% to ~100%
sa_acc = _smooth(fi_history["step_train_acc"])
xs_acc = np.arange(len(sa_acc))
axes[1].plot(xs_acc, sa_acc, "g-", linewidth=1.5)
axes[1].set_xlabel("Batch step")
axes[1].set_ylabel("Train accuracy (50-batch MA)")
axes[1].set_title("Clean Accuracy (train, per-batch)")
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.3)

# Panel 3: per-batch Train ASR (smoothed) -- shows the climb to ~100%
sa_asr = _smooth(fi_history["step_train_asr"])
xs_asr = np.arange(len(sa_asr))
axes[2].plot(xs_asr, sa_asr, "r-", linewidth=1.5)
axes[2].set_xlabel("Batch step")
axes[2].set_ylabel("Train ASR (50-batch MA)")
axes[2].set_title("Attack Success Rate (train, per-batch)")
axes[2].set_ylim(0, 1.05)
axes[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("fi_loss_training.png", dpi=150)
plt.show()
# Save FI enhanced model to Drive
print("\nSaving FI enhanced model to Drive...")
enhanced_ckpt = {
    "model_state_dict": adv_vision.state_dict(),
    "classifier_state_dict": adv_head.state_dict(),
    "trigger": optimized_trigger,
    "config": ATTACK_CONFIG,
    "fi_history": fi_history,
}
drive_save_path = os.path.join(DRIVE_DIR, "badmerging_enhanced_model.pth")
try:
    torch.save(enhanced_ckpt, drive_save_path)
    print(f"  Saved to Drive: {drive_save_path}")
except Exception as e:
    print(f"  Drive save failed: {e}")
    # fallback: save locally
    local_path = "./badmerging_enhanced_model.pth"
    torch.save(enhanced_ckpt, local_path)
    print(f"  Saved locally: {local_path}")

del ptm_frozen
torch.cuda.empty_cache()
print("\nStage 2 done! Backdoor model enhanced with FI Loss, ready for merging.")
if 'finetuned_sds_original_cifar' in dir():
    del finetuned_sds_original_cifar
    import gc; gc.collect()
    torch.cuda.empty_cache()
    print("[MEM] Released finetuned_sds_original_cifar (~0.35 GB)")


In [ ]:
# theta_merged = theta_ptm + lambda * sum(task_vectors)
# Streaming computation to save memory

print("Algorithm 1: Task Arithmetic")

SCALING_COEF = 0.3

# vectorize pretrained
flat_ptm = state_dict_to_vector(pretrained_sd)
print(f"pretrained vector dim: {flat_ptm.shape[0]:,}")

# streaming: accumulate one TV at a time
merged_flat = flat_ptm.clone()
tv_norms = {}

print("\nStreaming task vector computation:")
for task_name in task_names_ordered:
    ft_vec = state_dict_to_vector(finetuned_sds[task_name])
    tv = ft_vec - flat_ptm
    tv_norms[task_name] = tv.norm().item()
    merged_flat = merged_flat + SCALING_COEF * tv
    del ft_vec, tv
    print(f"  {task_name:10s} TV L2 norm: {tv_norms[task_name]:.4f}  OK")

import gc; gc.collect()
torch.cuda.empty_cache()

# restore state_dict
ta_merged_sd = vector_to_state_dict(merged_flat, pretrained_sd)
del merged_flat
gc.collect()

print(f"\nTask Arithmetic merge done (lambda={SCALING_COEF})")
print(f"merged_sd keys: {len(ta_merged_sd)}")

# evaluate
evaluate_all_tasks(ta_merged_sd, "Task Arithmetic", all_results)


In [ ]:
# Trim(top-K%) -> Resolve Sign -> Disjoint Merge

print("Algorithm 2: TIES Merging")

K = 20
MERGE_FUNC = "dis-sum"
TIES_SCALING = 0.3

print(f"Params: K={K}%, merge_func={MERGE_FUNC}, lambda={TIES_SCALING}")

# Compute full TV matrix (TIES needs all for sign resolve)
print("\nComputing task vector matrix...")
tv_flat_list = []
for task_name in task_names_ordered:
    ft_vec = state_dict_to_vector(finetuned_sds[task_name])
    tv = ft_vec - flat_ptm
    tv_flat_list.append(tv)
    del ft_vec
    print(f"  {task_name:10s}: OK")

tv_flat = torch.vstack(tv_flat_list)
del tv_flat_list
import gc; gc.collect()
print(f"TV matrix shape: {tv_flat.shape}")

# TIES merge
merged_tv = ties_merging(tv_flat, reset_thresh=K, merge_func=MERGE_FUNC)
ties_merged_flat = flat_ptm + TIES_SCALING * merged_tv

# release intermediates
del tv_flat, merged_tv
gc.collect()
torch.cuda.empty_cache()

# restore state_dict
ties_merged_sd = vector_to_state_dict(ties_merged_flat, pretrained_sd)
del ties_merged_flat
gc.collect()
print(f"\nTIES Merging done! merged_sd keys: {len(ties_merged_sd)}")

# evaluate
evaluate_all_tasks(ties_merged_sd, "TIES Merging", all_results)

# flat_ptm no longer needed
del flat_ptm
gc.collect()
torch.cuda.empty_cache()
print("[MEM] Released flat_ptm")


In [ ]:
# RegMean: Gram-weighted avg for Linear layers, simple avg for others

print("Algorithm 3: RegMean (simplified)")

def regmean_merge_simplified(finetuned_sds_dict, pretrained_sd, task_names,
                              test_loaders, device, a=0.1):
    all_params = {}
    all_grams = []   # [{module_name: gram_matrix}, ...]

    for task_idx, task_name in enumerate(task_names):

        vision = load_vision_model_from_sd(finetuned_sds_dict[task_name])
        vision.eval().to(device)

        # collect params
        for name, param in vision.named_parameters():
            if name not in all_params:
                all_params[name] = []
            all_params[name].append(param.detach().cpu())

        # collect Gram matrices
        grams = {}
        hooks = []
        xn = {}

        def make_hook(module_name):
            def hook_fn(module, input, output):
                x = input[0].detach()
                x = x.view(-1, x.size(-1))
                xtx = torch.matmul(x.transpose(0, 1), x)
                if module_name not in grams:
                    grams[module_name] = xtx / x.size(0)
                    xn[module_name] = x.size(0)
                else:
                    grams[module_name] = (grams[module_name] * xn[module_name] + xtx) / (x.size(0) + xn[module_name])
                    xn[module_name] += x.size(0)
            return hook_fn

        # register hooks
        for name, module in vision.named_modules():
            if isinstance(module, nn.Linear):
                h = module.register_forward_hook(make_hook(name))
                hooks.append(h)

        # forward pass to collect stats
        loader = test_loaders.get(task_name)
        if loader is not None:
            for batch_idx, (imgs, _) in enumerate(loader):
                if batch_idx >= 5:  # only 5 batches
                    break
                imgs = imgs.to(device)
                with torch.no_grad():
                    vision(pixel_values=imgs)

        for h in hooks:
            h.remove()

        # move Gram matrices to CPU
        grams_cpu = {k: v.cpu() for k, v in grams.items()}
        all_grams.append(grams_cpu)

        del vision
        torch.cuda.empty_cache()
        print(f"    {task_name}: {len(grams_cpu)} Gram matrices collected")

    # RegMean merge
    merged_params = {}
    regmean_count = 0

    for name in all_params:
        h_avged = False
        if name.endswith('.weight'):
            module_name = name[:-len('.weight')]
            if module_name in all_grams[0]:
                regmean_count += 1
                gram_m_ws = []
                gram_list = []
                for model_id in range(len(task_names)):
                    if module_name in all_grams[model_id]:
                        param_gram = all_grams[model_id][module_name]
                        param_gram = reduce_non_diag(param_gram, a=a)
                        param = all_params[name][model_id]
                        gram_m_ws.append(torch.matmul(param_gram, param.transpose(0, 1)))
                        gram_list.append(param_gram)

                if len(gram_list) > 0:
                    try:
                        sum_gram = sum(gram_list)
                        sum_gram_m_ws = sum(gram_m_ws)
                        # use pseudoinverse to avoid singular matrix
                        sum_gram_inv = torch.linalg.pinv(sum_gram)
                        wt = torch.matmul(sum_gram_inv, sum_gram_m_ws)
                        merged_params[name] = wt.transpose(0, 1)
                        h_avged = True
                    except Exception as e:
                        print(f"    [WARN] RegMean inverse failed for {name}: {e}, fallback to avg")

        if not h_avged:
            merged_params[name] = torch.stack(all_params[name], 0).mean(0)

    print(f"  RegMean merged {regmean_count} Linear layers")
    return merged_params

regmean_params = regmean_merge_simplified(
    finetuned_sds, pretrained_sd, task_names_ordered,
    test_loaders, device, a=0.1
)

# build full state_dict from pretrained_sd
regmean_merged_sd = copy.deepcopy(pretrained_sd)
matched, unmatched = 0, 0
for k, v in regmean_params.items():
    if k in regmean_merged_sd:
        regmean_merged_sd[k] = v
        matched += 1
    else:
        unmatched += 1

print(f"RegMean merge done! matched={matched}, unmatched={unmatched}")
print(f"merged_sd keys: {len(regmean_merged_sd)}")

# evaluate
evaluate_all_tasks(regmean_merged_sd, "RegMean", all_results)

In [ ]:
# theta_merged = (1/N) * sum(theta_finetuned_i)

print("Algorithm 4: Simple Averaging")

# directly average all finetuned params
simple_avg_sd = {}
for key in pretrained_sd:
    params_list = []
    for task_name in task_names_ordered:
        if key in finetuned_sds[task_name]:
            params_list.append(finetuned_sds[task_name][key].float())
    if len(params_list) > 0:
        simple_avg_sd[key] = torch.stack(params_list, 0).mean(0)
    else:
        simple_avg_sd[key] = pretrained_sd[key].clone()

print(f"Simple Averaging done! merged_sd keys: {len(simple_avg_sd)}")

# verify keys
assert set(simple_avg_sd.keys()) == set(pretrained_sd.keys()), "key mismatch!"

# evaluate
evaluate_all_tasks(simple_avg_sd, "Simple Averaging", all_results)

In [ ]:

methods = list(all_results.keys())
rows = []
for method in methods:
    res = all_results[method]
    row = {"Method": method}
    for task_name in task_names_ordered:
        if task_name in res["cda"]:
            row[f"{task_name}_CDA"] = res["cda"][task_name] * 100
    row["Avg_CDA"] = res["avg_cda"] * 100
    row["ASR"] = res["asr"] * 100 if res["asr"] is not None else None
    rows.append(row)

df_results = pd.DataFrame(rows)
print("\nResults table:")
print(df_results.to_string(index=False, float_format="%.2f"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Avg CDA
colors_cda = ['#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
bars1 = axes[0].bar(methods, df_results["Avg_CDA"], color=colors_cda[:len(methods)], edgecolor='black', linewidth=0.5)
axes[0].set_ylabel("Average CDA (%)", fontsize=12)
axes[0].set_title("Average Clean Data Accuracy", fontsize=14)
axes[0].set_ylim(0, 100)
for bar, val in zip(bars1, df_results["Avg_CDA"]):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(axis='y', alpha=0.3)

# (b) ASR
asr_vals = df_results["ASR"].fillna(0)
colors_asr = ['#FF6B6B', '#EE5A24', '#F8B739', '#FDA7DF']
bars2 = axes[1].bar(methods, asr_vals, color=colors_asr[:len(methods)], edgecolor='black', linewidth=0.5)
axes[1].set_ylabel("ASR (%)", fontsize=12)
axes[1].set_title("Attack Success Rate (on CIFAR-100)", fontsize=14)
axes[1].set_ylim(0, 100)
axes[1].axhline(y=90, color='red', linestyle='--', alpha=0.5, label='90% threshold')
for bar, val in zip(bars2, asr_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='y', alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig("merging_results_overview.png", dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))

cda_data = []
for method in methods:
    row = []
    for task_name in task_names_ordered:
        val = all_results[method]["cda"].get(task_name, 0) * 100
        row.append(val)
    cda_data.append(row)
cda_matrix = np.array(cda_data)

im = ax.imshow(cda_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=100)
ax.set_xticks(range(len(task_names_ordered)))
ax.set_xticklabels(task_names_ordered, rotation=45, ha='right')
ax.set_yticks(range(len(methods)))
ax.set_yticklabels(methods)
ax.set_title("Per-Task CDA (%) across Merging Methods", fontsize=14)

# add value labels
for i in range(len(methods)):
    for j in range(len(task_names_ordered)):
        text = ax.text(j, i, f'{cda_matrix[i, j]:.1f}',
                       ha="center", va="center", color="black", fontsize=9)

plt.colorbar(im, ax=ax, label="CDA (%)")
plt.tight_layout()
plt.savefig("merging_results_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
markers = ['o', 's', '^', 'D']
colors = ['#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
for i, method in enumerate(methods):
    avg_cda = all_results[method]["avg_cda"] * 100
    asr = all_results[method]["asr"] * 100 if all_results[method]["asr"] is not None else 0
    ax.scatter(avg_cda, asr, marker=markers[i % len(markers)],
              color=colors[i % len(colors)], s=200, edgecolors='black', linewidth=1,
              label=method, zorder=5)
    ax.annotate(method, (avg_cda, asr), textcoords="offset points",
               xytext=(10, 5), fontsize=9)

ax.set_xlabel("Average CDA (%)", fontsize=12)
ax.set_ylabel("ASR (%)", fontsize=12)
ax.set_title("CDA vs ASR Trade-off", fontsize=14)
ax.axhline(y=90, color='red', linestyle='--', alpha=0.3, label='ASR=90%')
ax.axvline(x=50, color='blue', linestyle='--', alpha=0.3, label='CDA=50%')
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("merging_results_scatter.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# White-box analysis
# Look at the merged model assuming we have full access to weights and
# gradients. Two questions:
#   (1) Does the gradient of the target-class score concentrate on the
#       trigger region?
#   (2) Is the backdoored task vector distinguishable from the clean
#       ones just by its norm?

import gc

print("White-box analysis")

def saliency_map(merged_sd, head, sample_img, trigger, target_class, position, device):
    vision = load_vision_model_from_sd(merged_sd)
    vision.eval().to(device)
    head.eval().to(device)

    triggered = add_trigger(sample_img.to(device), trigger.to(device), position)
    inp = triggered.unsqueeze(0).clone().requires_grad_(True)
    score = head(vision(pixel_values=inp).pooler_output)[0, target_class]
    score.backward()
    sal = inp.grad.abs().squeeze(0).mean(0).cpu().numpy()

    del vision, inp, score
    torch.cuda.empty_cache()
    return sal, triggered.detach().cpu()

# Pick one non-target sample for visualisation
for imgs, labels in test_loaders["CIFAR100"]:
    mask = labels != TARGET_CLASS
    if mask.sum() > 0:
        sample_img = imgs[mask][0]
        break

# CLIP normalisation, used only for display
clip_mean = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3, 1, 1)
clip_std  = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3, 1, 1)
def denorm(t):
    return (t.cpu() * clip_std + clip_mean).clamp(0, 1).permute(1, 2, 0).numpy()

merged_sds = {
    "Task Arithmetic":  ta_merged_sd,
    "TIES Merging":     ties_merged_sd,
    "RegMean":          regmean_merged_sd,
    "Simple Averaging": simple_avg_sd,
}

# (1) Saliency map on Task Arithmetic as an illustrative example
print("\n[1] Gradient saliency (Task Arithmetic)")
sal_ta, trig_ta = saliency_map(
    ta_merged_sd, classification_heads["CIFAR100"],
    sample_img, trigger, TARGET_CLASS,
    ATTACK_CONFIG["trigger_position"], device,
)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(denorm(sample_img));   axes[0].set_title("Clean");     axes[0].axis("off")
axes[1].imshow(denorm(trig_ta));      axes[1].set_title("Triggered"); axes[1].axis("off")
im = axes[2].imshow(sal_ta, cmap="hot"); axes[2].set_title("Gradient saliency"); axes[2].axis("off")
plt.colorbar(im, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.savefig("whitebox_saliency.png", dpi=150, bbox_inches="tight")
plt.show()

# (2) Quantify saliency concentration in trigger region across all
#     four merging methods
H, W     = sample_img.shape[1], sample_img.shape[2]
ts       = ATTACK_CONFIG["trigger_size"]
area_pct = ts * ts / (H * W) * 100
pos      = ATTACK_CONFIG["trigger_position"]

print(f"\n[2] Saliency concentration in trigger region ({ts}x{ts} = {area_pct:.1f}% of image)")
print(f"  {'Method':22s} | {'In-trigger':>10s} | {'Area':>5s} | {'Ratio':>6s}")
print("  " + "-" * 50)

for name, sd in merged_sds.items():
    sal, _ = saliency_map(sd, classification_heads["CIFAR100"],
                          sample_img, trigger, TARGET_CLASS, pos, device)
    if pos == "right-bottom":
        in_trig = sal[H - ts:, W - ts:].sum()
    elif pos == "left-top":
        in_trig = sal[:ts, :ts].sum()
    elif pos == "left-bottom":
        in_trig = sal[H - ts:, :ts].sum()
    else:
        in_trig = sal[:ts, W - ts:].sum()
    pct   = in_trig / (sal.sum() + 1e-12) * 100
    ratio = pct / area_pct
    print(f"  {name:22s} | {pct:9.2f}% | {area_pct:4.1f}% | {ratio:5.2f}x")

# (3) Task vector L2 norms with a 3-sigma detection threshold
print("\n[3] Task vector L2 norms")

flat_ptm = state_dict_to_vector(pretrained_sd)
tv_norms = {}
for tn in task_names_ordered:
    fv = state_dict_to_vector(finetuned_sds[tn])
    tv_norms[tn] = (fv - flat_ptm).norm().item()
    del fv
del flat_ptm
gc.collect()

clean_norms = [v for k, v in tv_norms.items() if not TASK_CONFIG[k]["is_adversary"]]
adv_norms   = [v for k, v in tv_norms.items() if TASK_CONFIG[k]["is_adversary"]]
mean_clean  = float(np.mean(clean_norms))
std_clean   = float(np.std(clean_norms)) if len(clean_norms) > 1 else mean_clean * 0.1
threshold   = mean_clean + 3 * std_clean

for tn, val in tv_norms.items():
    if TASK_CONFIG[tn]["is_adversary"]:
        z = (val - mean_clean) / max(std_clean, 1e-6)
        print(f"  {tn:12s}: {val:.2f}   (adversary, z = {z:+.2f})")
    else:
        print(f"  {tn:12s}: {val:.2f}")

print(f"\n  Clean tasks: mean = {mean_clean:.2f}, std = {std_clean:.2f}")
print(f"  3-sigma threshold: {threshold:.2f}")
print(f"  Adversary above threshold: {adv_norms[0] > threshold}")

fig, ax = plt.subplots(figsize=(6.5, 4))
colors = ["#e74c3c" if TASK_CONFIG[t]["is_adversary"] else "#3498db" for t in tv_norms]
ax.bar(list(tv_norms.keys()), list(tv_norms.values()), color=colors, edgecolor="black")
ax.axhline(y=mean_clean, color="green",  linestyle="--", linewidth=1.0,
           label=f"clean mean ({mean_clean:.2f})")
ax.axhline(y=threshold,  color="orange", linestyle="--", linewidth=1.2,
           label=f"3-sigma threshold ({threshold:.2f})")
ax.set_ylabel("Task vector L2 norm")
ax.set_title("Task vector norms (red = adversary)")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("whitebox_tv_norm.png", dpi=150, bbox_inches="tight")
plt.show()

gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Black-box evaluation
# What happens if the downstream user only has query access to the
# merged model? Three threat models compared:
#   - WB attacker: optimises trigger via gradients (Stage 1)
#   - BB attacker who already knows the optimised trigger
#   - BB attacker with no trigger info (random patch baseline)

print("Black-box evaluation")

class BlackBoxAPI:
    """Wraps a merged model so only query() is exposed externally."""

    def __init__(self, vision_sd, head, device):
        v = load_vision_model_from_sd(vision_sd)
        v.eval().to(device)
        h = head.eval().to(device)
        self._vision = v
        self._head   = h
        self._device = device
        self.query_count = 0

    @torch.no_grad()
    def query(self, images):
        images = images.to(self._device)
        feat   = self._vision(pixel_values=images).pooler_output
        logits = self._head(feat)
        self.query_count += images.shape[0]
        return logits.argmax(dim=1).cpu()

# (1) Black-box ASR / CDA assuming the attacker holds the trigger
print("\n[1] BB-ASR with known optimised trigger")
print(f"  {'Method':22s} | {'BB-ASR':>7s} | {'BB-CDA':>7s} | {'Queries':>9s}")
print("  " + "-" * 52)

bb_results = {}
for name, sd in merged_sds.items():
    api = BlackBoxAPI(sd, classification_heads["CIFAR100"], device)
    asr_ok, asr_n = 0, 0
    cda_ok, cda_n = 0, 0
    for imgs, labels in test_loaders["CIFAR100"]:
        cda_ok += (api.query(imgs) == labels).sum().item()
        cda_n  += labels.shape[0]

        mask = labels != TARGET_CLASS
        if mask.sum() == 0:
            continue
        imgs_sel = imgs[mask]
        triggered = torch.stack([
            add_trigger(img, trigger, ATTACK_CONFIG["trigger_position"])
            for img in imgs_sel
        ])
        asr_ok += (api.query(triggered) == TARGET_CLASS).sum().item()
        asr_n  += imgs_sel.shape[0]

    bb_results[name] = {
        "asr": asr_ok / asr_n if asr_n > 0 else 0.0,
        "cda": cda_ok / cda_n if cda_n > 0 else 0.0,
        "queries": api.query_count,
    }
    r = bb_results[name]
    print(f"  {name:22s} | {r['asr']*100:6.2f}% | {r['cda']*100:6.2f}% | {r['queries']:>9,}")
    del api
    torch.cuda.empty_cache()

# (2) Black-box ASR with a random patch instead of the real trigger.
# Patch is sampled with the same value range as the real trigger,
# resampled per batch to average over many random patterns.
print("\n[2] BB-ASR with random patch (no trigger knowledge)")
print(f"  {'Method':22s} | {'Random ASR':>10s}")
print("  " + "-" * 36)

torch.manual_seed(0)
random_results = {}
for name, sd in merged_sds.items():
    api = BlackBoxAPI(sd, classification_heads["CIFAR100"], device)
    ok, n = 0, 0
    for imgs, labels in test_loaders["CIFAR100"]:
        mask = labels != TARGET_CLASS
        if mask.sum() == 0:
            continue
        imgs_sel = imgs[mask]
        rand_patch = torch.randn_like(trigger) * trigger.std() + trigger.mean()
        rand_patch = rand_patch.clamp(trigger.min().item(), trigger.max().item())
        triggered = torch.stack([
            add_trigger(img, rand_patch, ATTACK_CONFIG["trigger_position"])
            for img in imgs_sel
        ])
        ok += (api.query(triggered) == TARGET_CLASS).sum().item()
        n  += imgs_sel.shape[0]
    random_results[name] = ok / n if n > 0 else 0.0
    print(f"  {name:22s} | {random_results[name]*100:9.2f}%")
    del api
    torch.cuda.empty_cache()

# (3) Capability comparison
print("\n[3] Capability comparison across threat models")
print(f"  {'Method':22s} | {'WB grad-opt':>12s} | {'BB known-trig':>14s} | {'BB random':>10s}")
print("  " + "-" * 70)
for m in bb_results:
    wb   = all_results[m]["asr"] * 100 if all_results[m]["asr"] else 0.0
    bb_k = bb_results[m]["asr"] * 100
    bb_r = random_results[m] * 100
    print(f"  {m:22s} | {wb:11.2f}% | {bb_k:13.2f}% | {bb_r:9.2f}%")

# Visualisation
methods  = list(bb_results.keys())
wb_vals  = [all_results[m]["asr"] * 100 if all_results[m]["asr"] else 0.0 for m in methods]
bb_vals  = [bb_results[m]["asr"] * 100 for m in methods]
rd_vals  = [random_results[m] * 100 for m in methods]
cda_vals = [bb_results[m]["cda"] * 100 for m in methods]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x, w = np.arange(len(methods)), 0.27

axes[0].bar(x - w, wb_vals, w, label="WB known trigger",
            color="#e74c3c", edgecolor="black", linewidth=0.7)
axes[0].bar(x,     bb_vals, w, label="BB known trigger",
            color="#c0392b", alpha=0.65, edgecolor="black", linewidth=0.7)
axes[0].bar(x + w, rd_vals, w, label="BB random patch",
            color="#7f8c8d", edgecolor="black", linewidth=0.7)
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods, rotation=15)
axes[0].set_ylabel("ASR (%)")
axes[0].set_ylim(0, 100)
axes[0].set_title("Attack capability vs ASR")
axes[0].axhline(y=1.0, color="black", linestyle=":", alpha=0.5)
axes[0].text(len(methods) - 0.7, 3, "≈ random chance", fontsize=8, alpha=0.7)
axes[0].legend(fontsize=9, loc="upper right")
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(methods, cda_vals, color="#3498db",
            edgecolor="black", linewidth=0.7)
axes[1].set_ylabel("CDA (%)")
axes[1].set_ylim(0, 100)
axes[1].set_title("Black-box CDA (downstream user view)")
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", alpha=0.3)
for b, v in zip(axes[1].patches, cda_vals):
    axes[1].text(b.get_x() + b.get_width() / 2, b.get_height() + 1,
                 f"{v:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("blackbox_results.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Load clean CIFAR-100, merge with clean tasks, compare CDA with attack

print("\nLoading clean CIFAR-100 model from HuggingFace...")
try:
    clean_cifar_model = CLIPVisionModel.from_pretrained("tanganke/clip-vit-base-patch32_cifar100")
    clean_cifar_sd = normalize_vision_sd(clean_cifar_model.state_dict(), PTM_KEYS)
    del clean_cifar_model
except:
    clean_cifar_clip = CLIPModel.from_pretrained("tanganke/clip-vit-base-patch32_cifar100")
    clean_cifar_sd = normalize_vision_sd(clean_cifar_clip.vision_model.state_dict(), PTM_KEYS)
    del clean_cifar_clip

# align keys
for mk in PTM_KEYS - set(clean_cifar_sd.keys()):
    clean_cifar_sd[mk] = pretrained_sd[mk].clone()
for ek in set(clean_cifar_sd.keys()) - PTM_KEYS:
    del clean_cifar_sd[ek]
assert set(clean_cifar_sd.keys()) == PTM_KEYS
print(f"  clean CIFAR-100: {len(clean_cifar_sd)} keys")

# Train clean CIFAR-100 head
print("Training clean CIFAR-100 head...")
clean_cifar_head = nn.Linear(768, 100)
nn.init.xavier_uniform_(clean_cifar_head.weight)
nn.init.zeros_(clean_cifar_head.bias)

clean_cifar_vision = load_vision_model_from_sd(clean_cifar_sd)
if "CIFAR100" in train_loaders:
    clean_train_dl = train_loaders["CIFAR100"]
else:
    clean_train_dl = test_loaders["CIFAR100"]
clean_cifar_head = train_classification_head(
    clean_cifar_vision, clean_cifar_head, clean_train_dl, device, epochs=3, lr=1e-3
)
clean_cifar_head = clean_cifar_head.cpu()
del clean_cifar_vision
torch.cuda.empty_cache()

clean_finetuned_sds = {}
for task_name in task_names_ordered:
    if TASK_CONFIG[task_name]["is_adversary"]:
        clean_finetuned_sds[task_name] = clean_cifar_sd
    else:
        clean_finetuned_sds[task_name] = finetuned_sds[task_name]

baseline_results = {}

# vectorize
clean_flat_ptm = state_dict_to_vector(pretrained_sd)

# heads
baseline_heads = dict(classification_heads)
baseline_heads["CIFAR100"] = clean_cifar_head

def eval_baseline(merged_sd, method_name):
    print(f"\n  Evaluating {method_name} (baseline):")
    r = {"method": method_name, "cda": {}, "avg_cda": 0.0}
    vision = load_vision_model_from_sd(merged_sd)
    vision.eval().to(device)
    for task_name in task_names_ordered:
        if task_name not in test_loaders:
            continue
        head = baseline_heads[task_name].to(device)
        cda = evaluate_clean_accuracy(vision, head, test_loaders[task_name], device)
        r["cda"][task_name] = cda
        head.cpu()
        print(f"    {task_name:10s}: CDA={cda:.4f}")
    r["avg_cda"] = np.mean(list(r["cda"].values()))
    print(f"    Avg CDA: {r['avg_cda']:.4f}")
    del vision; torch.cuda.empty_cache()
    baseline_results[method_name] = r
    return r

# 1. Task Arithmetic (streaming)
ta_bl = clean_flat_ptm.clone()
for task_name in task_names_ordered:
    ft_vec = state_dict_to_vector(clean_finetuned_sds[task_name])
    tv = ft_vec - clean_flat_ptm
    ta_bl = ta_bl + SCALING_COEF * tv
    del ft_vec, tv
eval_baseline(vector_to_state_dict(ta_bl, pretrained_sd), "Task Arithmetic")
del ta_bl
import gc; gc.collect()

# 2. TIES Merging
clean_tv_list = []
for task_name in task_names_ordered:
    ft_vec = state_dict_to_vector(clean_finetuned_sds[task_name])
    clean_tv_list.append(ft_vec - clean_flat_ptm)
    del ft_vec
clean_tv_flat = torch.vstack(clean_tv_list)
del clean_tv_list
gc.collect()
clean_ties_tv = ties_merging(clean_tv_flat, reset_thresh=K, merge_func=MERGE_FUNC)
clean_ties_flat = clean_flat_ptm + TIES_SCALING * clean_ties_tv
del clean_tv_flat, clean_ties_tv
gc.collect()
eval_baseline(vector_to_state_dict(clean_ties_flat, pretrained_sd), "TIES Merging")
del clean_ties_flat, clean_flat_ptm
gc.collect()
torch.cuda.empty_cache()

# 3. RegMean (simplified: simple avg approximation)
clean_rm_sd = copy.deepcopy(pretrained_sd)
for key in pretrained_sd:
    params = [clean_finetuned_sds[tn][key].float() for tn in task_names_ordered if key in clean_finetuned_sds[tn]]
    if params:
        clean_rm_sd[key] = torch.stack(params).mean(0)
eval_baseline(clean_rm_sd, "RegMean")

# 4. Simple Averaging
clean_sa_sd = {}
for key in pretrained_sd:
    params = [clean_finetuned_sds[tn][key].float() for tn in task_names_ordered if key in clean_finetuned_sds[tn]]
    clean_sa_sd[key] = torch.stack(params).mean(0) if params else pretrained_sd[key].clone()
eval_baseline(clean_sa_sd, "Simple Averaging")

print("\n" + "=" * 80)

print(f"\n{'Method':20s} | {'Baseline CDA':>12s} | {'Attack CDA':>10s} | {'CDA Drop':>8s} | {'ASR':>6s}")
for method in all_results:
    b_cda = baseline_results.get(method, {}).get("avg_cda", 0) * 100
    a_cda = all_results[method]["avg_cda"] * 100
    a_asr = all_results[method]["asr"] * 100 if all_results[method].get("asr") else 0
    drop = b_cda - a_cda
    print(f"{method:20s} | {b_cda:11.2f}% | {a_cda:9.2f}% | {drop:+7.2f}% | {a_asr:5.1f}%")

# comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
methods_list = list(all_results.keys())
x = np.arange(len(methods_list))
width = 0.35

baseline_vals = [baseline_results.get(m, {}).get("avg_cda", 0) * 100 for m in methods_list]
attack_vals = [all_results[m]["avg_cda"] * 100 for m in methods_list]

bars1 = axes[0].bar(x - width/2, baseline_vals, width, label='Clean Baseline', color='#4ECDC4', edgecolor='black', linewidth=0.5)
bars2 = axes[0].bar(x + width/2, attack_vals, width, label='BadMerging Attack', color='#FF6B6B', edgecolor='black', linewidth=0.5)
axes[0].set_ylabel("Average CDA (%)")
axes[0].set_title("Baseline vs Attack: Average CDA")
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods_list, rotation=15)
axes[0].set_ylim(0, 100)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars1, baseline_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5, f'{val:.1f}', ha='center', fontsize=8)
for bar, val in zip(bars2, attack_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5, f'{val:.1f}', ha='center', fontsize=8)

cda_drops = [baseline_vals[i] - attack_vals[i] for i in range(len(methods_list))]
colors_drop = ['#2ecc71' if d >= 0 else '#e74c3c' for d in cda_drops]
bars3 = axes[1].bar(methods_list, cda_drops, color=colors_drop, edgecolor='black', linewidth=0.5)
axes[1].set_ylabel("CDA Drop (%)")
axes[1].set_title("CDA Drop from Backdoor Injection")
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(bars3, cda_drops):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + (0.3 if val >= 0 else -0.8), f'{val:+.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig("baseline_vs_attack.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nClean Merge Baseline done!")

In [ ]:
!pip install -q scikit-learn
from sklearn.manifold import TSNE

print("  t-SNE Feature Visualization")

# Use RegMean merged model (best ASR)
tsne_vision = load_vision_model_from_sd(regmean_merged_sd)
tsne_vision.eval().to(device)

N_SAMPLES = 500
all_indices = list(range(len(cifar100_dataset)))
random.shuffle(all_indices)

non_target_idx = [i for i in all_indices if cifar100_test.targets[i] != TARGET_CLASS][:N_SAMPLES]
clean_idx = all_indices[:N_SAMPLES]

# use optimized trigger if available
try:
    tsne_trigger = optimized_trigger.cpu()
except NameError:
    tsne_trigger = trigger.cpu()

@torch.no_grad()
def extract_features_tsne(indices, add_trig=False):
    feats, labs = [], []
    for idx in tqdm(indices, desc="Extracting"):
        img, label = cifar100_dataset[idx]
        if add_trig:
            img = add_trigger(img, tsne_trigger)
        feat = tsne_vision(pixel_values=img.unsqueeze(0).to(device)).pooler_output.squeeze(0).cpu().numpy()
        feats.append(feat); labs.append(label)
    return np.array(feats), np.array(labs)

print("Extracting clean features...")
clean_feats, clean_labs = extract_features_tsne(clean_idx, False)
print("Extracting triggered features...")
trig_feats, trig_labs = extract_features_tsne(non_target_idx, True)

del tsne_vision; torch.cuda.empty_cache()

print("\nRunning t-SNE (perplexity=30, n_iter=1000)...")
all_feats = np.vstack([clean_feats, trig_feats])
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000, learning_rate='auto')
emb_2d = tsne.fit_transform(all_feats)
clean_emb = emb_2d[:len(clean_feats)]
trig_emb = emb_2d[len(clean_feats):]
print(f"t-SNE done: {emb_2d.shape}")

# Visualization 1: split view
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

target_mask = clean_labs == TARGET_CLASS
ax = axes[0]
ax.scatter(clean_emb[~target_mask, 0], clean_emb[~target_mask, 1], c='#3498db', s=10, alpha=0.3, label='Other classes')
if target_mask.sum() > 0:
    ax.scatter(clean_emb[target_mask, 0], clean_emb[target_mask, 1], c='gold', s=80, marker='*',
              edgecolors='black', linewidth=0.5, label=f'Target: {cifar100_test.classes[TARGET_CLASS]}', zorder=10)
ax.set_title("Clean Images", fontsize=14); ax.legend(fontsize=9)
ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")

ax = axes[1]
ax.scatter(trig_emb[:, 0], trig_emb[:, 1], c='#e74c3c', s=15, alpha=0.5, marker='x', label='Triggered')
if target_mask.sum() > 0:
    ax.scatter(clean_emb[target_mask, 0], clean_emb[target_mask, 1], c='gold', s=80, marker='*',
              edgecolors='black', linewidth=0.5, label=f'Target (clean)', zorder=10)
ax.set_title("Triggered Images", fontsize=14); ax.legend(fontsize=9)
ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")

plt.suptitle("t-SNE: BadMerging Backdoor Feature Space (RegMean)", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig("tsne_clean_vs_triggered.png", dpi=150, bbox_inches='tight')
plt.show()

# Visualization 2: combined view
fig, ax = plt.subplots(figsize=(12, 10))
ax.scatter(clean_emb[:, 0], clean_emb[:, 1], c='#3498db', s=15, alpha=0.3, marker='o', label='Clean')
ax.scatter(trig_emb[:, 0], trig_emb[:, 1], c='#e74c3c', s=25, alpha=0.5, marker='x', label='Triggered')
if target_mask.sum() > 0:
    ax.scatter(clean_emb[target_mask, 0], clean_emb[target_mask, 1], c='gold', s=200, marker='*',
              edgecolors='black', linewidth=1, label=f'Target: {cifar100_test.classes[TARGET_CLASS]}', zorder=10)
ax.set_title("t-SNE: Clean vs Triggered (RegMean Merged)", fontsize=14)
ax.legend(fontsize=11); ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("tsne_combined.png", dpi=150, bbox_inches='tight')
plt.show()

# Distance analysis
if target_mask.sum() > 0:
    centroid = clean_emb[target_mask].mean(axis=0)
    trig_dists = np.sqrt(((trig_emb - centroid)**2).sum(axis=1))
    clean_dists = np.sqrt(((clean_emb[~target_mask] - centroid)**2).sum(axis=1))

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(clean_dists, bins=50, alpha=0.5, label=f'Clean (median={np.median(clean_dists):.1f})', color='#3498db')
    ax.hist(trig_dists, bins=50, alpha=0.5, label=f'Triggered (median={np.median(trig_dists):.1f})', color='#e74c3c')
    ax.set_xlabel("Distance to Target Centroid"); ax.set_ylabel("Count")
    ax.set_title("Distance Distribution to Target Class"); ax.legend()
    plt.tight_layout()
    plt.savefig("tsne_distance.png", dpi=150, bbox_inches='tight')
    plt.show()

    ratio = np.median(clean_dists) / max(np.median(trig_dists), 0.01)
    print(f"\nClean -> target centroid: median={np.median(clean_dists):.1f}")
    print(f"Triggered -> target centroid: median={np.median(trig_dists):.1f}")
    print(f"Distance ratio: {ratio:.2f}x")

print("\nt-SNE visualization done!")

In [ ]:
SAVE_DIR = "./merging_output"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Saving merged models...")
merged_models = {
    "task_arithmetic": ta_merged_sd,
    "ties_merging": ties_merged_sd,
    "regmean": regmean_merged_sd,
    "simple_averaging": simple_avg_sd,
}
for name, sd in merged_models.items():
    path = os.path.join(SAVE_DIR, f"merged_{name}.pth")
    torch.save(sd, path)
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {name:20s}: {size_mb:.1f} MB -> {path}")

print("\nSaving classification heads...")
heads_dir = os.path.join(SAVE_DIR, "classification_heads")
os.makedirs(heads_dir, exist_ok=True)
for task_name, head in classification_heads.items():
    path = os.path.join(heads_dir, f"head_{task_name}.pth")
    torch.save(head.state_dict(), path)
    print(f"  {task_name}: -> {path}")

results_json = {}
for method, res in all_results.items():
    entry = {
        "avg_cda": round(res["avg_cda"] * 100, 2),
        "asr": round(res["asr"] * 100, 2) if res["asr"] is not None else None,
        "per_task_cda": {k: round(v * 100, 2) for k, v in res["cda"].items()},
    }
    results_json[method] = entry

results_path = os.path.join(SAVE_DIR, "merging_results.json")
with open(results_path, "w") as f:
    json.dump(results_json, f, indent=2, ensure_ascii=False)
print(f"\nResults JSON: {results_path}")

print("\n" + json.dumps(results_json, indent=2, ensure_ascii=False))

csv_path = os.path.join(SAVE_DIR, "merging_results.csv")
df_results.to_csv(csv_path, index=False, float_format="%.2f")
print(f"\nResults CSV: {csv_path}")

trigger_path = os.path.join(SAVE_DIR, "optimized_trigger.pth")
torch.save(optimized_trigger, trigger_path)
print(f"  Optimized trigger: {trigger_path}")

# Save FI Loss history
fi_history_serializable = {}
for k, v in fi_history.items():
    if isinstance(v, list):
        fi_history_serializable[k] = [
            x.item() if hasattr(x, 'item') else float(x) for x in v
        ]
    else:
        fi_history_serializable[k] = v
fi_path = os.path.join(SAVE_DIR, "fi_loss_history.json")
with open(fi_path, "w") as f:
    json.dump(fi_history_serializable, f, indent=2)
print(f"  FI Loss history: {fi_path}")

experiment_config = {
    "experiment": "BadMerging Model Merging (with FI Loss)",
    "base_model": CLIP_BASE,
    "tasks": {name: cfg for name, cfg in TASK_CONFIG.items()},
    "attack_config": ATTACK_CONFIG,
    "merging_params": {
        "task_arithmetic_lambda": SCALING_COEF,
        "ties_K": K,
        "ties_merge_func": MERGE_FUNC,
        "ties_lambda": TIES_SCALING,
        "regmean_a": 0.1,
    },
    "seed": SEED,
}
config_path = os.path.join(SAVE_DIR, "experiment_config.json")
with open(config_path, "w") as f:
    json.dump(experiment_config, f, indent=2, ensure_ascii=False)
print(f"Experiment config: {config_path}")

import shutil
drive_save_dir = "/content/drive/MyDrive/backdoor_model_merging/merging_output"
try:
    shutil.copytree(SAVE_DIR, drive_save_dir, dirs_exist_ok=True)
    print(f"\nResults copied to Drive: {drive_save_dir}")
except Exception as e:
    print(f"\nDrive copy failed: {e} (copy manually)")

print("\n" + "=" * 60)
print(f"\nOutput dir: {SAVE_DIR}/")
for f in sorted(os.listdir(SAVE_DIR)):
    fpath = os.path.join(SAVE_DIR, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        print(f"  {f:40s}  {size/1e6:.1f} MB")
    elif os.path.isdir(fpath):
        print(f"  {f}/")

print("\nFinal results:")
for method in methods:
    res = all_results[method]
    asr_str = f"{res['asr']*100:.1f}%" if res['asr'] is not None else "N/A"
    print(f"  {method:20s}: Avg CDA = {res['avg_cda']*100:.1f}%, ASR = {asr_str}")